In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# important workflow libraries:
import torch, gc, os, sys, tiktoken, json, types
from torch.utils.data import Dataset
from transformers import PretrainedConfig
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
import random

In [3]:
# storing paths:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [4]:
# adding paths to system:
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [5]:
# getting GPU if available:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
# importing modules:
from gpt_model import GPTModel
from config import GPTConfig
CONFIG = GPTConfig()

In [7]:
model = GPTModel(CONFIG)

GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [8]:
# from tiktoken_tokenizer import TiktokenTokenizer

# # Create and save tokenizer
# tokenizer = TiktokenTokenizer(encoding_name="gpt2")
# tokenizer.save_pretrained("/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model")
# print("Tokenizer saved successfully!")

In [9]:
torch.cuda.empty_cache()
gc.collect()

60

In [10]:
# model.load_state_dict(torch.load("/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth"))
# print("Weights_loaded successfully.")

Weights_loaded successfully.


In [12]:
# model.save_pretrained(
#     "/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model",
#     safe_serialization=False  # ADD THIS
# )
# print("Model saved!")

GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwar

Model saved!


In [15]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 436.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 939.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 634.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [ ]:
# to load jsonl and convert data into required formate:
def load_and_convert_data(input_file):
    data = []
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if "conversations" in item and len(item["conversations"]) >= 2:
                data.append({
                    "messages": [
                        {"role": "user", "content": item["conversations"][0]},
                        {"role": "assistant", "content": item["conversations"][1]}
                    ]
                })
    return data

In [ ]:
instruction_data = load_and_convert_data("/content/drive/MyDrive/llm_from_scratch/datasets/instruction_data_lima.jsonl")

In [ ]:
print(len(instruction_data),  "\n")
instruction_data[44]

1030 



{'messages': [{'role': 'user',
   'content': "Is there a Git for data? The key improvement I'd want is to Diff/Merge more intelligently. e.g. in CSV rather than line vs line comparison, it would do cell vs cell.\nAnd ordering is usually not significant, e.g. rows in a CSV, whereas Git does care and presents the user with 'conflicts'."},
  {'role': 'assistant',
   'content': 'Please note that ```git``` has two configuration commands:\n```git config filter..clean\ngit config filter..smudge\n```\nThe ```clean``` filter can be used e.g. to sort all records in a CSV file except the top row, thus making re-ordering irrelevant.\nQuoting ```gitattributes(5)``` :\n\nA filter driver consists of a clean command and a smudge command, either of which can be left unspecified. Upon checkout, when the smudge command is specified, the command is fed the blob object from its standard input, and its standard output is used to update the worktree file. Similarly, the clean command is used to convert the c

In [ ]:
greeting_and_qa_data = json.load(open("/content/drive/MyDrive/llm_from_scratch/datasets/greeting_and_qa.json"))

In [ ]:
print(len(greeting_and_qa_data),  "\n")
greeting_and_qa_data[44]

452 



{'messages': [{'role': 'user',
   'content': "What's the background of your developer?"},
  {'role': 'assistant',
   'content': 'My developer, Muhammad Toqeer Yasir, has strong background in artificial intelligence, machine learning, deep learning, and natural language processing. He created me from the ground up in 2025 as a comprehensive AI project.'}]}

In [ ]:
with open('/content/drive/MyDrive/llm_from_scratch/datasets/instruction_data_alpaca_gpt4.json', 'r') as f:
  alpaca_data = json.load(f)
print(len(alpaca_data),  "\n")
alpaca_data[44]

1100 



{'instruction': 'Reverse the order of the given phrase.',
 'input': 'Moon and stars',
 'output': 'Stars and moon'}

In [ ]:
with open('/content/drive/MyDrive/llm_from_scratch/datasets/orca_agentinstruct_10k_balanced.json', 'r') as f:
    orca_instruction_data = json.load(f)

for item in orca_instruction_data:
    if isinstance(item.get('messages'), str):
        item['messages'] = json.loads(item['messages'])

orca_instruction_data[0]

{'messages': [{'role': 'system', 'content': ''},
  {'role': 'user',
   'content': "Create an outline for a presentation that includes five engaging and informative slides to test the audience's understanding of the concepts discussed in your talk, such as p-value functions, compatibility intervals, and the misconceptions about p-values. Each slide should present a multiple-choice question related to these topics. Prepare talking points that explain the correct answers and why they are correct, ensuring the explanations are clear and suitable for an educational presentation."},
  {'role': 'assistant',
   'content': 'Title: Understanding P-Values and Statistical Inference\n\nSlide 1: Introduction to P-Values\n- Multiple-Choice Question: What does a p-value indicate in hypothesis testing?\n  A) The probability that the null hypothesis is true\n  B) The probability of observing the data, or something more extreme, if the null hypothesis is true\n  C) The probability that the alternative hy

In [ ]:
random.seed(44)
orca_instruction_data = random.sample(orca_instruction_data, 7418)
len(orca_instruction_data)

7418

## **Combining datasets to create a single one and shuffle it.**

In [ ]:
merged_data = greeting_and_qa_data + orca_instruction_data + instruction_data + alpaca_data

In [ ]:
random.seed(44)
random.shuffle(merged_data)

In [ ]:
print(len(merged_data))
merged_data[44]

10000


{'messages': [{'role': 'user',
   'content': "I am 21 years old and living in a large city in Germany where smalltalk in local markets is not a common thing.\nA new cashier joined my local food shop. She’s always at the checkout and never doing stuff like sorting products or cleaning the floor where I could actually ask her out. I am quite new to relationships, but the signs she gave me are promising.\nMy question is how I can ask for her number, or ask her out for coffee while she is only sitting at the checkout? I mean there are always like 5 people before and after me, and I think it would be awkward if we are changing numbers while customers are waiting behind us. Or even worse if I read the signs wrong and she rejects me? Since the store is just 5 min away from my place I visit regularly and don't want to leave a bad impression there."},
  {'role': 'assistant',
   'content': "Asking her out directly seems like a bad idea.\nInstead, in these situations, I've found something that wo

In [ ]:
class ConvertDataIntoTokenIds(torch.utils.data.Dataset):
    def __init__(self, data, max_length=512):
        self.data = data
        self.tokenizer = tiktoken.get_encoding('gpt2')
        self.max_length = max_length
        self.eot_token = 50256

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        user_message= assistant= ""

        if "instruction" in item:
            text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        elif item["messages"][0]["role"] == "system":
            user_msg = item["messages"][1]["content"]
            assistant_msg = item["messages"][2]["content"]
            text = f"User: {user_msg}\nAssistant: {assistant_msg}"

        else:
            user_msg = item["messages"][0]["content"]
            assistant_msg = item["messages"][1]["content"]
            text = f"User: {user_msg}\nAssistant: {assistant_msg}"

        tokens = self.tokenizer.encode(text)

        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        if len(tokens) < self.max_length:
            padding = [self.eot_token] * (self.max_length - len(tokens))
            tokens = tokens + padding

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(tokens, dtype=torch.long)
        }

In [ ]:
# creating dataset
train_dataset = ConvertDataIntoTokenIds(merged_data)

In [ ]:
train_dataset[244]

{'input_ids': tensor([12982,    25, 13610,   281,  3053,  7124,  1923,   329,   257, 38435,
          5735, 13948,  2139,   326, 11330,   262,  4034,   286,  1719,  1963,
         34900,  1104,   329,  4273,  4393,   508,  2740,  3614,  3594,    13,
           383,  1923,   815,  2291,    25,   198,   198,    16,    13,   317,
          2426,  1627,   326, 23007,   262,  3241,   286,  4273,  4393,   508,
           743,   307,  6476,  3303, 14725,   618,  6095, 38435,  1337,    13,
           198,   198,    17,    13,  1052,  4756,   326, 48004,  4340,   351,
           262,  6459,   286, 18088,   329,  6639, 17252,   290,   262,  2087,
          8722,   286,  3303, 14725,    11,   840,   870,   606,   326,   674,
          2139,  3769,  5887, 38435,  5608,   351,   262,  6829,   286,  4779,
          2024,    13,   198,   198,    18,    13,   317,  2665,   326, 20718,
           674, 38435,  6154,    11, 40318,   511,  2694,   284, 37220, 16500,
          3303, 14725,   290,  2148, 28

In [ ]:
# attaching config to model
model.config = GPTConfig()

In [ ]:
#  attaching monkey patch for forword function:
def hf_forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
    batch_size, seq_len = input_ids.shape

    with torch.amp.autocast('cuda', enabled=True):
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

    loss = None
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    return (loss, logits) if loss is not None else logits

model.forward = hf_forward.__get__(model, type(model))

In [ ]:
model.unload()
print('Model unloaded successfully.')

Model unloaded successfully.


In [ ]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [ ]:
# applying lora addapters for reducing model size:
def setup_lora_model(model):

    target_modules = [
        "W_query", "W_key", "W_value",
        "out_proj",
        "out_head"
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# training arguments for peft:
import transformers
transformers.logging.set_verbosity_info()
training_args = TrainingArguments(
    # output & saving
    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",

    save_strategy="no",

    # training hyperparameters
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    weight_decay=0.01,

    # optimization
    optim="adafactor",
    lr_scheduler_type="linear",
    warmup_ratio=0.05,

    # precision & performance
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,

    # logging & monitoring
    logging_strategy="steps",
    logging_steps=5,
    disable_tqdm=False,
    report_to="none"
)

PyTorch: setting up devices


In [ ]:
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
# creating trianer:
os.environ["WANDB_DISABLED"] = "true"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
# clearing memory:
torch.cuda.empty_cache()
gc.collect()

print("Started training...")
trainer.train()
print("Training completed!")

***** Running training *****
  Num examples = 10,000
  Num Epochs = 2
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 1,250
  Number of trainable parameters = 3,361,416


Started training...


Step,Training Loss
5,13.779300
10,14.874800
15,15.803200
20,12.985900
25,12.892300
30,15.632000
35,15.525000
40,15.392400
45,15.078000
50,15.076000


In [ ]:
# def test_model_fast(model, input_text="", max_new_tokens=1024):
#     encoding = tiktoken.get_encoding('gpt2')
#     eot_token = encoding.eot_token

#     # Tokenize
#     input_ids = encoding.encode(input_text)
#     input_tensor = torch.tensor([input_ids]).to(device)
#     generated = input_tensor

#     model.eval()
#     with torch.no_grad():
#         for i in range(max_new_tokens):
#             # Forward pass - your model returns logits directly
#             logits = model(generated)

#             # Get last token logits and apply temperature
#             next_token_logits = logits[0, -1, :] / 0.1

#             # Convert to probabilities and sample
#             probs = torch.softmax(next_token_logits, dim=-1)
#             next_token = torch.multinomial(probs, num_samples=1)

#             # Stop at EOT token
#             if next_token.item() == eot_token:
#                 break

#             # Append new token
#             generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

#     # Return only the generated part (after input)
#     full_output = encoding.decode(generated[0].tolist())
#     return full_output[len(input_text):].replace('<|endoftext|>', '').strip()

In [ ]:
# # Test with your model
# print("🧪 Testing the fine-tuned model:\n")
# model = model.to(device)

# test1 = test_model_fast(
#     model,
#     input_text="hello!",
# )
# print(f"\n{test1}\n")

🧪 Testing the fine-tuned model:



KeyboardInterrupt: 

In [ ]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/finetuned_model.pth")

# print("✅ Saved unified model with instruction knowledge")

✅ Saved unified model with instruction knowledge
